# Folding story with more pixels

A follow-up to [`story.ipynb`](story.ipynb): same idea (one printed square that shows a new picture at each stage of folding a crane), aimed at its biggest weakness. The story here: **home planet (flat) → rocket (bird base) → another planet (crane).**

## The problem: the crane gets very few pixels

DeepFloyd draws in two stages: a 64 × 64 image that decides the composition, then a 256 × 256 upscale that adds detail. Everything is drawn on the whole sheet, and folding decides what share of the sheet each stage shows. That share is fixed by geometry, so printing bigger doesn't help:

| | Stage 1 (64 × 64) | 128 × 128 | Stage 2 (256 × 256) |
|---|---|---|---|
| Bird base paper pixels | 848 | 3,392 | 13,572 |
| Crane paper pixels | **489** | 1,925 | 7,681 |

In `story.ipynb`, the crane's picture is composed in those 489 pixels (its body is about 16 × 22), and stage 2 can only sharpen what's there. That's why busy prompts came out unreadable on the crane.

## Round 1: two fixes for resolution

1. **Stage 2 redraws the stage pictures.** Stage 2 is told how far it may depart from stage 1 by a *noise level*: IF's default, 250, means "mostly keep it". We gave the folded views a higher level (700), so stage 2 would compose the rocket and the planet at 256 pixels. **It changed almost nothing.** Every pixel a folded view shows is also in the flat view, which keeps 250 and half the vote, so stage 1's picture stays put. Dropped.
2. **Stage 1 at 128 × 128.** The model is convolutional, so it runs at 128 and every view gets four times the pixels. **This worked for the crane:** the planet became a clear round shape on its body, and the flat landscape got much richer. But the bird base got two small rockets instead of one big one. The model draws objects at the size it learned at 64, so a tall 128 canvas fits two.

Round 1 also showed a prompt problem. The folded prompts asked for "a plain light background", so the model left most of each folded view pale. On the flat sheet, that pale paper became white patches framing the red rockets: the opposite of hiding them.

## Round 2: prompts that fill the paper

The silhouette already separates a folded picture from the table, so the subject doesn't need an empty background. Round 2 drops "plain light background" and compares four prompt sets, then varies the resolution and the backdrop for one of them:

| Prompt set | The rocket's prompt |
|---|---|
| **alone** | a red and white rocket ship flying upward |
| **close-up** | a close-up of a red and white rocket ship filling the frame |
| **over the scene** | a red rocket flying over forests and rivers, seen from above |
| **made of the scene's colors** | a red rocket launching in a cloud of fiery orange exhaust |

The last set is my best guess. Each subject's surroundings are made of the flat scene's colors (orange exhaust like the autumn trees, blue swirling clouds like the rivers), so they fill the paper and blend into the landscape when it's flat, without asking the model to put a rocket in a landscape where it doesn't belong.

The **backdrop** is the plain color drawn around each folded view's silhouette. It was light gray (0.85), which nudged the model toward pale paper too. Round 2 uses mid gray (0.5), with one run at 0.85 to compare.

## Why only the crane

The plan was to also try Yakko (a traditional figure that shows more paper than the crane, in more pieces). But a story needs the in-between stages, and Flat-Folder's Yakko has none we can use. A search over every subset of its creases found that, keeping each crease's mountain or valley, only the full model and the flat sheet (with its corner tips folded in) fold flat. Its real folding sequence must pass through stages where some creases are folded the other way and later reversed. To add Yakko, we'd need its folding sequence (fold one and note the steps), then build each stage's crease pattern from that, as `make_stages.py` does for the crane.

## Setup

Clones [rath-hy/folded-illusions](https://github.com/rath-hy/folded-illusions/tree/main/fold-map) on Colab, or uses the local files if you run this from that folder.

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("story.py"):
    if not os.path.exists("folded-illusions"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/folded-illusions.git"], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "directions", "decal", "illusion", "story"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from illusion import generate, print_page
from story import least_shared_sides, load_stages, shared_paper, stage_views, story_images

stages, models = load_stages()                 # bird base, crane
SIDES = least_shared_sides(models)
names = [name for name, _, _, _ in stages]
for name, side, share in zip(names, SIDES, shared_paper(models, SIDES)):
    print(f"{name} ({side}): {share:.0%} of its visible paper also shows at the other stage")

## Install and log in (needs a GPU)

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the license for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece
from huggingface_hub import notebook_login
notebook_login()

## The story

Choices that held up from round 1:

- **Home planet as an aerial view, not a globe.** The flat sheet is where the other two pictures hide, so it should be busy everywhere. A globe on black space would leave large plain areas where any hidden shape stands out (and black prints muddy).
- **The flat palette includes the later pictures' colors.** Orange autumn trees give the red rocket somewhere to hide; blue rivers and purple-gray shadows do the same for the planet.
- **A red rocket pointing up, for the bird base,** which is a tall, narrow diamond.
- **A purple planet for the crane,** whose body diamond is where a round shape reads best. Gray would print dull.

Each prompt set below is one prompt per stage, in folding order (bird base, crane). Add your own sets to compare them.

In [ ]:
STYLE = "a watercolor painting of "
FLAT_PROMPT = STYLE + "an aerial view of green forests, orange autumn trees, gray mountains and winding blue rivers"
PROMPT_SETS = {
    "alone": [
        "a red and white rocket ship flying upward",
        "a purple planet with rings",
    ],
    "close-up": [
        "a close-up of a red and white rocket ship filling the frame",
        "a close-up of a purple planet with rings filling the frame",
    ],
    "over the scene": [
        "a red rocket flying over forests and rivers, seen from above",
        "a purple planet with rings above forests and rivers",
    ],
    "scene's colors": [
        "a red rocket launching in a cloud of fiery orange exhaust",
        "a purple planet surrounded by swirling blue clouds",
    ],
}
PROMPT_SETS = {k: [STYLE + p for p in v] for k, v in PROMPT_SETS.items()}
assert all(len(v) == len(stages) for v in PROMPT_SETS.values()), f"one prompt per stage: {names}"
# Other ideas: "a silver rocket with orange flames", "Saturn", "a cratered gray moon" (your original idea);
# style "a colorful children's book illustration of " (bolder shapes, which survive printing better).

## Encode the prompts

The text encoder is loaded onto the GPU, used once, and deleted to save memory. Every prompt set is encoded together with the flat prompt. **Rerun this cell whenever you change the prompts.** The first run downloads about 11 GB.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
embeds = {}                                    # prompt set -> (negative, positive), flat first, then each stage
for key, prompts in PROMPT_SETS.items():
    pos, neg = encoder.encode_prompt([FLAT_PROMPT] + prompts)
    embeds[key] = (neg, pos)

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("encoded:", ", ".join(embeds))

## Load the image models

Once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

## Experiments

Each experiment sets:

- `prompts`: which prompt set.
- `res1`: the stage-1 resolution. 64 is what IF was trained at; 128 gave the crane a readable planet in round 1 but split the rocket in two; 96 is in between.
- `background`: the backdrop around each folded silhouette, from 0 (black) to 1 (white).

The first four compare prompt sets. The last two change one thing each from the "scene's colors" run. On a T4, expect several minutes for all six.

In [ ]:
EXPERIMENTS = {
    "1. alone":                  dict(prompts="alone",          res1=128, background=0.5),
    "2. close-up":               dict(prompts="close-up",       res1=128, background=0.5),
    "3. over the scene":         dict(prompts="over the scene", res1=128, background=0.5),
    "4. scene's colors":         dict(prompts="scene's colors", res1=128, background=0.5),
    "5. scene's colors, 96":     dict(prompts="scene's colors", res1=96,  background=0.5),
    "6. scene's colors, light":  dict(prompts="scene's colors", res1=128, background=0.85),
}
FOLD_WEIGHT = 0.5
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

views = {}                                     # FoldViews per resolution, made once
def views_at(res):
    if res not in views:
        views[res] = stage_views(stages, models, SIDES, res)
    return views[res]

def run(ex, seed):
    return generate(stage_1, stage_2, views_at(ex["res1"]), views_at(256), embeds[ex["prompts"]],
                    guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, background=ex["background"], seed=seed)

results = {}
for label, ex in EXPERIMENTS.items():
    start = time.time()
    results[label] = run(ex, SEED)
    print(f"{label}: {time.time() - start:.0f} s")

## Compare

One row per experiment: the printed sheet, then each stage folded from it with the decal simulator (independent of the maps used to make the sheet). The second figure shows the stage-1 sheets, where the composition is decided.

What to look for:

- **Does the rocket read on the bird base, and the planet on the crane?**
- **Does the flat sheet look like one landscape,** or do rocket and planet shapes show through? This is what round 2 is about.
- **Is there one rocket or two?** If 96 keeps one rocket and the planet still reads, it's the better resolution.

In [ ]:
cols = ["flat (home planet)"] + [f"{name} ({side})" for name, side in zip(names, SIDES)]
fig, axes = plt.subplots(len(results), len(cols), figsize=(4 * len(cols), 4 * len(results)), squeeze=False)
for i, (row, (label, (_, final))) in enumerate(zip(axes, results.items())):
    for ax, img, col in zip(row, story_images(final, stages, models, SIDES), cols):
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(col if i == 0 else "")
    row[0].set_ylabel(label, fontsize=13)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4.3))
for ax, (label, (first, _)) in zip(axes, results.items()):
    ax.imshow(first, interpolation="nearest")
    ax.set_title(f"{label}\nstage 1, {first.shape[0]} × {first.shape[0]}", fontsize=10)
    ax.axis("off")
plt.show()

## Try more seeds

Seeds matter as much as settings, so once one experiment looks promising, run it on a few seeds before judging it.

In [ ]:
PICK = "4. scene's colors"
SEEDS = [1, 2, 3]

for seed in SEEDS:
    results[f"{PICK}, seed {seed}"] = run(EXPERIMENTS[PICK], seed)
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 4.3))
    for ax, img, col in zip(axes, story_images(results[f"{PICK}, seed {seed}"][1], stages, models, SIDES), cols):
        ax.imshow(img); ax.set_title(f"seed {seed}: {col}"); ax.axis("off")
    plt.show()

## Print the best one

Saves the sheet, and an A4 PDF with it at 200 mm, centered, with a thin gray cut line. Print at **actual size** (not "fit to page"), cut along the line, and fold with the **printed side out**. The bird base is held with its wing flaps up.

In [ ]:
BEST = "4. scene's colors"                   # any key of results, e.g. "4. scene's colors, seed 2"

name = "planets_" + "_".join("".join(c for c in w if c.isalnum()) for w in BEST.split())   # e.g. planets_4_scenes_colors_seed_2
img = print_page(results[BEST][1], name + "_a4.pdf")   # 200 mm square at 300 dpi
img.save(name + ".png", dpi=(300, 300))
print("saved", name + ".png", "and", name + "_a4.pdf")
try:
    from google.colab import files
    files.download(name + "_a4.pdf")
except ImportError:
    pass

## Notes

- **Per-view stage-2 noise levels stay in `generate`** (`noise_level=[flat, bird base, crane]`), though round 1 found they barely matter while the flat view holds every pixel at 250.
- **The bigger fix, for later: mixed scales.** Generate the whole sheet at high resolution, so the folded views get their pixels one to one, and show the flat view a downscaled copy at 64. The catch is the noise: averaging blocks of pixels keeps the picture but shrinks the noise, which the model isn't trained for. Topping it up with fresh noise fixes the flat view's input, but then its prediction only covers the coarse image, and the fine detail in hidden areas needs a prediction from somewhere else. *Factorized Diffusion* (Geng, Park, Owens, 2024) deals with a similar split between coarse and fine. Getting this right would be a real research contribution.
- **A bigger stage 1 model.** `IF-I-L-v1.0` (900M parameters) instead of `IF-I-M-v1.0` (400M) composes better and may fit a T4. Swap the name in the load cell.

## References

- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- D. Geng, I. Park, A. Owens. [Factorized Diffusion: Perceptual Illusions by Noise Decomposition](https://dangeng.github.io/factorized_diffusion/). *ECCV*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.
- H. A. Akitaya, E. D. Demaine, J. S. Ku. [Computing Flat-Folded States](https://erikdemaine.org/papers/FlatFolder_OSME2024/). *OSME*, 2024.